# MET Concealed-Probe Rejection Rates by Prompt Distribution

This notebook plots the concealed-probe MET frontier for the `api_met_kl_s150_w20_h20_u40` adapter. Each figure shows the mean MET rejection rate at alpha 0.05 across split seeds, with one standard-deviation error bars. The x-axis reports the additional concealed prompt mass as a percentage of the original MET prompt distribution size.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 240
plt.rcParams["axes.titleweight"] = "semibold"
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10


## Configuration

The notebook reads the compact concealed-probe summary artifact and writes publication-style PNGs to `images/`.


In [2]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SUMMARY_PATH = REPO_ROOT / "artifacts" / "model_equality_section5" / "concealed_probe_frontier_api_met_kl_s150_w20_h20_u40_seed0_9" / "summary.csv"
IMAGE_DIR = REPO_ROOT / "images"

SUITE_LABELS = {
    "humaneval": "HumanEval",
    "ultrachat": "UltraChat",
    "wikipedia_en": "Wikipedia",
}
SUITE_ORDER = ["wikipedia_en", "ultrachat", "humaneval"]
OUTPUT_FILENAMES = {
    "wikipedia_en": "met_concealed_probe_wikipedia_rejection_rate.png",
    "ultrachat": "met_concealed_probe_ultrachat_rejection_rate.png",
    "humaneval": "met_concealed_probe_humaneval_rejection_rate.png",
}

LINE_COLOR = "#7A5195"
LINE_MARKER_SIZE = 8
LINE_WIDTH = 2.8
ERRORBAR_CAPSIZE = 5
ERRORBAR_LINEWIDTH = 1.6
SCORE_LABEL_FONT_SIZE = 10
GRID_COLOR = "#E5E7EB"


def save_figure(fig, filename: str) -> Path:
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    path = IMAGE_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=300)
    return path


print(f"Summary path: {SUMMARY_PATH}")
print(f"Image dir: {IMAGE_DIR}")


Summary path: /Users/angadkalra/Desktop/uni_code/robust-auditing/artifacts/model_equality_section5/concealed_probe_frontier_api_met_kl_s150_w20_h20_u40_seed0_9/summary.csv
Image dir: /Users/angadkalra/Desktop/uni_code/robust-auditing/images


## Load Summary Data

`summary.csv` contains one aggregate row per concealed level and prompt distribution. The `0%` point is the public-anchor adapter result; the other points aggregate the split-seeded concealed-probe runs.


In [3]:
summary_df = pd.read_csv(SUMMARY_PATH)
plot_df = (
    summary_df[summary_df["suite"].isin(SUITE_ORDER)]
    .copy()
    .sort_values(["suite", "concealed_level"])
)
plot_df["suite_label"] = plot_df["suite"].map(SUITE_LABELS)
plot_df["concealed_level_label"] = plot_df["concealed_level"].map(lambda value: f"{int(value)}%")
plot_df[[
    "concealed_level",
    "suite_label",
    "seed_count",
    "mean_rejection_rate_alpha_0_05",
    "std_rejection_rate_alpha_0_05",
    "row_type",
]]


,concealed_level,suite_label,seed_count,mean_rejection_rate_alpha_0_05,std_rejection_rate_alpha_0_05,row_type
0,0,HumanEval,1,0.130,0.000000,public_anchor
3,25,HumanEval,10,0.089,0.020790,split
6,50,HumanEval,10,0.091,0.030350,split
9,75,HumanEval,10,0.068,0.032931,split
12,100,HumanEval,10,0.092,0.026998,split
1,0,UltraChat,1,0.320,0.000000,public_anchor
4,25,UltraChat,10,0.262,0.056529,split
7,50,UltraChat,10,0.446,0.080305,split
10,75,UltraChat,10,0.542,0.072847,split
13,100,UltraChat,10,0.663,0.082603,split


## Per-Distribution Rejection Rate Figures

The three figures use the same style convention as the BOLD baseline audit plots, but omit a legend because each panel contains only one series. All panels use the same 0.0-0.8 y-axis and mark the 0.5 suite-failure threshold.


In [4]:
REJECTION_RATE_Y_LIMITS = (0.0, 0.8)
FAILURE_THRESHOLD = 0.5


def plot_suite_rejection_rate(suite: str) -> Path:
    group = plot_df[plot_df["suite"] == suite].sort_values("concealed_level")
    if group.empty:
        raise ValueError(f"Missing suite in summary: {suite}")

    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.errorbar(
        group["concealed_level"],
        group["mean_rejection_rate_alpha_0_05"],
        yerr=group["std_rejection_rate_alpha_0_05"],
        marker="o",
        markersize=LINE_MARKER_SIZE,
        linewidth=LINE_WIDTH,
        color=LINE_COLOR,
        ecolor=LINE_COLOR,
        elinewidth=ERRORBAR_LINEWIDTH,
        capsize=ERRORBAR_CAPSIZE,
        capthick=ERRORBAR_LINEWIDTH,
    )
    for _, point in group.iterrows():
        ax.annotate(
            f"{point['mean_rejection_rate_alpha_0_05']:.2f}",
            xy=(point["concealed_level"], point["mean_rejection_rate_alpha_0_05"]),
            xytext=(0, 12),
            textcoords="offset points",
            ha="center",
            fontsize=SCORE_LABEL_FONT_SIZE,
            color="#1F2937",
            clip_on=False,
        )

    suite_label = SUITE_LABELS[suite]
    ax.set_title(f"{suite_label}: MET rejection rate under concealed prompts", pad=10)
    ax.set_xlabel("Extra concealed prompts (% of original MET suite)")
    ax.set_ylabel("Mean rejection rate")
    ax.set_xticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"])
    ax.set_xlim(-4, 104)
    ax.set_ylim(*REJECTION_RATE_Y_LIMITS)
    ax.axhline(FAILURE_THRESHOLD, color="0.35", linestyle="--", linewidth=1.5)
    ax.grid(axis="y", color=GRID_COLOR)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax)

    fig.tight_layout()
    path = save_figure(fig, OUTPUT_FILENAMES[suite])
    plt.close(fig)
    return path


figure_paths = [plot_suite_rejection_rate(suite) for suite in SUITE_ORDER]
figure_paths


[PosixPath('/Users/angadkalra/Desktop/uni_code/robust-auditing/images/met_concealed_probe_wikipedia_rejection_rate.png'),
 PosixPath('/Users/angadkalra/Desktop/uni_code/robust-auditing/images/met_concealed_probe_ultrachat_rejection_rate.png'),
 PosixPath('/Users/angadkalra/Desktop/uni_code/robust-auditing/images/met_concealed_probe_humaneval_rejection_rate.png')]